In [ ]:
def main(datasources, start_date, end_date):
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]
    LOOKBACK_DAYS = max(0, 10)
    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    start_date_text = str(start_date)
    end_date_text = str(end_date)
    calendar_buffer = (
        max(LOOKBACK_DAYS + 7, (LOOKBACK_DAYS * 7 + 4) // 5 + 10)
        if LOOKBACK_DAYS
        else 0
    )
    query_start_date = (
        (start_ts - pd.Timedelta(days=calendar_buffer)).strftime("%Y-%m-%d")
        if calendar_buffer
        else start_date_text
    )

    sql = f"""
    WITH cte_bar1m AS (
        SELECT
            date,
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            (CASE WHEN ABS((((((ABS((CASE WHEN ABS((((((((bid_num_orders1) + (bid_num_orders2))) + (bid_num_orders3))) + (1e-08)))) > 1e-12 THEN (((((bid_volume1) + (bid_volume2))) + (bid_volume3))) / (((((((bid_num_orders1) + (bid_num_orders2))) + (bid_num_orders3))) + (1e-08))) ELSE NULL END))) + (ABS((CASE WHEN ABS((((((((ask_num_orders1) + (ask_num_orders2))) + (ask_num_orders3))) + (1e-08)))) > 1e-12 THEN (((((ask_volume1) + (ask_volume2))) + (ask_volume3))) / (((((((ask_num_orders1) + (ask_num_orders2))) + (ask_num_orders3))) + (1e-08))) ELSE NULL END))))) + (1e-08)))) > 1e-12 THEN ((((CASE WHEN ABS((((((((bid_num_orders1) + (bid_num_orders2))) + (bid_num_orders3))) + (1e-08)))) > 1e-12 THEN (((((bid_volume1) + (bid_volume2))) + (bid_volume3))) / (((((((bid_num_orders1) + (bid_num_orders2))) + (bid_num_orders3))) + (1e-08))) ELSE NULL END)) - ((CASE WHEN ABS((((((((ask_num_orders1) + (ask_num_orders2))) + (ask_num_orders3))) + (1e-08)))) > 1e-12 THEN (((((ask_volume1) + (ask_volume2))) + (ask_volume3))) / (((((((ask_num_orders1) + (ask_num_orders2))) + (ask_num_orders3))) + (1e-08))) ELSE NULL END)))) / (((((ABS((CASE WHEN ABS((((((((bid_num_orders1) + (bid_num_orders2))) + (bid_num_orders3))) + (1e-08)))) > 1e-12 THEN (((((bid_volume1) + (bid_volume2))) + (bid_volume3))) / (((((((bid_num_orders1) + (bid_num_orders2))) + (bid_num_orders3))) + (1e-08))) ELSE NULL END))) + (ABS((CASE WHEN ABS((((((((ask_num_orders1) + (ask_num_orders2))) + (ask_num_orders3))) + (1e-08)))) > 1e-12 THEN (((((ask_volume1) + (ask_volume2))) + (ask_volume3))) / (((((((ask_num_orders1) + (ask_num_orders2))) + (ask_num_orders3))) + (1e-08))) ELSE NULL END))))) + (1e-08))) ELSE NULL END) AS minute_factor
        FROM {bar1m}
    ),
    cte_window AS (
        SELECT
            trading_day,
            instrument,
            (last(minute_factor ORDER BY date)) * 1 AS base_factor
        FROM cte_bar1m
        GROUP BY instrument, trading_day
    ),
    cte_postprocess AS (
        SELECT
            trading_day,
            instrument,
            CASE
                WHEN COUNT(base_factor) OVER (PARTITION BY instrument ORDER BY trading_day ROWS BETWEEN 9 PRECEDING AND CURRENT ROW) >= 10
                THEN base_factor - AVG(base_factor) OVER (PARTITION BY instrument ORDER BY trading_day ROWS BETWEEN 9 PRECEDING AND CURRENT ROW)
                ELSE NULL
            END AS factor
        FROM cte_window
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        factor
    FROM cte_postprocess
    ORDER BY date, instrument
    """

    df = dai.query(
        sql,
        filters={"date": [query_start_date, end_date_text]},
        compression=True,
    ).df()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    df["factor"] = pd.to_numeric(df["factor"], errors="coerce")
    df = df[df["factor"].notna() & (df["factor"].abs() < float("inf"))]
    df = df[(df["date"] >= start_ts.normalize()) & (df["date"] <= end_ts.normalize())]

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date_text, end_date_text]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"], errors="coerce").dt.normalize()
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)
    df = pd.merge(df, stk_pool, how="inner", on=["date", "instrument"])
    df = df.dropna(subset=["date", "instrument", "factor"])
    return df[["date", "instrument", "factor"]].drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)
